In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/household_power_consumption.txt', sep=';')
df.shape

C:\Users\harsh\AppData\Local\Temp\ipykernel_5216\3239296682.py:4: DtypeWarning: Columns (0: Global_active_power, 1: Global_reactive_power, 2: Voltage, 3: Global_intensity, 4: Sub_metering_1, 5: Sub_metering_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/household_power_consumption.txt', sep=';')


(2075259, 9)

In [2]:
df = df.replace('?', np.nan)
df.isnull().sum()

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

In [3]:
cols_to_convert = ['Global_active_power', 'Global_reactive_power', 'Voltage',
                    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2']

for col in cols_to_convert:
    df[col] = df[col].astype(float)

df.dtypes

Date                         str
Time                         str
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

In [4]:
df['timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,timestamp
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


In [5]:
df = df.dropna()
df.shape

(2049280, 10)

In [6]:
df = df.sort_values('timestamp').reset_index(drop=True)

df['date'] = df['timestamp'].dt.date
df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()
df['is_weekend'] = df['timestamp'].dt.dayofweek >= 5

df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,timestamp,date,year,month,day,hour,day_of_week,is_weekend
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00,2006-12-16,2006,12,16,17,Saturday,True
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,2006-12-16,2006,12,16,17,Saturday,True
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,2006-12-16,2006,12,16,17,Saturday,True
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,2006-12-16,2006,12,16,17,Saturday,True
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00,2006-12-16,2006,12,16,17,Saturday,True


In [7]:
hourly_df = df.resample('h', on='timestamp').agg({
    'Global_active_power': 'mean',
    'Global_reactive_power': 'mean',
    'Voltage': 'mean',
    'Global_intensity': 'mean',
    'Sub_metering_1': 'sum',
    'Sub_metering_2': 'sum',
    'Sub_metering_3': 'sum'
}).reset_index()

hourly_df.shape

(34589, 8)

In [8]:
hourly_df['date'] = hourly_df['timestamp'].dt.date
hourly_df['year'] = hourly_df['timestamp'].dt.year
hourly_df['month'] = hourly_df['timestamp'].dt.month
hourly_df['hour'] = hourly_df['timestamp'].dt.hour
hourly_df['day_of_week'] = hourly_df['timestamp'].dt.day_name()
hourly_df['is_weekend'] = hourly_df['timestamp'].dt.dayofweek >= 5

hourly_df.head()

,timestamp,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,date,year,month,hour,day_of_week,is_weekend
0,2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,19.0,607.0,2006-12-16,2006,12,17,Saturday,True
1,2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,403.0,1012.0,2006-12-16,2006,12,18,Saturday,True
2,2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,86.0,1001.0,2006-12-16,2006,12,19,Saturday,True
3,2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.0,1007.0,2006-12-16,2006,12,20,Saturday,True
4,2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,25.0,1033.0,2006-12-16,2006,12,21,Saturday,True


In [9]:
hourly_df.to_csv('../data/processed/hourly_energy.csv', index=False)
print("Saved successfully!")

OSError: Cannot save file into a non-existent directory: '..\data\processed'

In [10]:
import os
os.makedirs('../data/processed', exist_ok=True)

In [11]:
hourly_df.to_csv('../data/processed/hourly_energy.csv', index=False)
print("Saved successfully!")

Saved successfully!
